# 🎓 Face Recognition Training & Evaluation (YOLOv8 + LBPH)
ระบบเทรนและประเมินผลจดจำใบหน้า:
1. **Multi-Cascade Face Detection:** ใช้ Haar Cascade หลายตัว + Fallback ตรง
2. **CLAHE Preprocessing:** ปรับสมดุลแสง + ลด Noise
3. **Data Augmentation:** หมุน ±5°, พลิกซ้ายขวา, ปรับแสง
4. **LBPH Training:** ฝึกสอนโมเดลและบันทึก `.yml`
5. **Config Export:** บันทึก label_dict + threshold ลง `model_config.json`
6. **F1-Score Threshold Optimization:** เลือก threshold ที่ดีที่สุดด้วย F1

In [ ]:
# 1. เชื่อมต่อ Google Drive (สำหรับ Google Colab)
import os

try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("✅ เชื่อมต่อ Google Drive สำเร็จ")
    IS_COLAB = True
except ImportError:
    print("ℹ️ รันใน Local Environment")
    IS_COLAB = False

In [ ]:
# 2. ติดตั้งและนำเข้าไลบรารี
import subprocess
import sys

def install_if_missing(package, import_name=None):
    try:
        __import__(import_name or package)
    except ImportError:
        print(f"กำลังติดตั้ง {package}...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package, "-q"])

install_if_missing("opencv-contrib-python", "cv2")
install_if_missing("ultralytics")

import os
import json
import cv2
import cv2.face
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from ultralytics import YOLO
from datetime import datetime

print(f"✅ OpenCV Version: {cv2.__version__}")

In [ ]:
# 3. ฟังก์ชัน Preprocessing (สร้าง CLAHE ครั้งเดียว)
CLAHE = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
TARGET_SIZE = (120, 120)

def preprocess_face(face_gray, target_size=TARGET_SIZE):
    """ปรับแต่งภาพใบหน้าเพื่อความแม่นยำสูงสุด"""
    face_resized = cv2.resize(face_gray, target_size, interpolation=cv2.INTER_CUBIC)
    face_clahe = CLAHE.apply(face_resized)
    face_denoised = cv2.GaussianBlur(face_clahe, (3, 3), 0)
    return face_denoised

print("✅ ฟังก์ชัน Preprocessing พร้อมใช้งาน")

In [ ]:
# 4. โหลด Dataset ด้วย Multi-Cascade Detection + Fallback

# --- ค้นหาโฟลเดอร์ Dataset ---
possible_paths = [
    'dataset', './dataset', '../dataset',
    '/content/drive/MyDrive/Project_Ai/dataset',
    '/content/drive/My Drive/Project_Ai/dataset',
    os.path.join(os.getcwd(), 'dataset')
]

DATASET_PATH = None
for p in possible_paths:
    if os.path.exists(p):
        DATASET_PATH = p
        break

if DATASET_PATH is None:
    raise FileNotFoundError("❌ ไม่พบโฟลเดอร์ dataset กรุณาตรวจสอบพาธ")

print(f"📂 ใช้ Dataset จาก: {DATASET_PATH}")

# --- โหลด Haar Cascades หลายตัว ---
cascade_paths = [
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml',
    cv2.data.haarcascades + 'haarcascade_frontalface_alt2.xml',
    cv2.data.haarcascades + 'haarcascade_profileface.xml',
]

cascades = []
for cp in cascade_paths:
    c = cv2.CascadeClassifier(cp)
    if not c.empty():
        cascades.append(c)
        print(f"  ✅ โหลด {os.path.basename(cp)} สำเร็จ")
    else:
        print(f"  ⚠️ ไม่สามารถโหลด {os.path.basename(cp)}")

if len(cascades) == 0:
    raise RuntimeError("❌ ไม่สามารถโหลด Haar Cascade ได้เลย")

face_cascade = cascades[0]  # ใช้ตัวหลักสำหรับ reference

def detect_faces_multi(gray_img, min_size=(30, 30)):
    """ตรวจจับใบหน้าด้วยหลาย cascade แล้วรวมผลลัพธ์"""
    all_faces = []

    # ลอง cascade ตัวแรก (frontalface_default) ก่อน
    for cascade in cascades:
        faces = cascade.detectMultiScale(
            gray_img, scaleFactor=1.1, minNeighbors=4, minSize=min_size
        )
        if len(faces) > 0:
            all_faces.extend(faces.tolist())

    if len(all_faces) == 0:
        # Fallback: ลดความเข้มงวดลง
        for cascade in cascades:
            faces = cascade.detectMultiScale(
                gray_img, scaleFactor=1.05, minNeighbors=3, minSize=(20, 20)
            )
            if len(faces) > 0:
                all_faces.extend(faces.tolist())
                break

    # ลบ bounding box ที่ซ้อนทับกัน (Non-Maximum Suppression แบบง่าย)
    if len(all_faces) <= 1:
        return all_faces

    unique_faces = []
    used = set()
    for i, (x1, y1, w1, h1) in enumerate(all_faces):
        if i in used:
            continue
        best = (x1, y1, w1, h1)
        best_area = w1 * h1
        for j, (x2, y2, w2, h2) in enumerate(all_faces):
            if j <= i or j in used:
                continue
            # คำนวณ overlap
            ox = max(0, min(x1+w1, x2+w2) - max(x1, x2))
            oy = max(0, min(y1+h1, y2+h2) - max(y1, y2))
            overlap = ox * oy
            min_area = min(w1*h1, w2*h2)
            if min_area > 0 and overlap / min_area > 0.5:
                used.add(j)
                if w2*h2 > best_area:
                    best = (x2, y2, w2, h2)
                    best_area = w2*h2
        unique_faces.append(best)
    return unique_faces

# --- สกัดใบหน้าจาก Dataset ---
faces_raw = []
labels_raw = []
label_dict = {}

person_names = sorted([
    d for d in os.listdir(DATASET_PATH)
    if os.path.isdir(os.path.join(DATASET_PATH, d))
])
print(f"\nพบรายชื่อทั้งหมด {len(person_names)} คน: {person_names}")

for current_id, person_name in enumerate(person_names):
    label_dict[current_id] = person_name
    person_path = os.path.join(DATASET_PATH, person_name)

    if not os.path.exists(person_path):
        print(f"  ⚠️ ข้ามโฟลเดอร์ที่ไม่พบ: {person_path}")
        continue

    count = 0
    skipped = 0

    for image_name in sorted(os.listdir(person_path)):
        if not image_name.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
            continue

        image_path = os.path.join(person_path, image_name)
        img = cv2.imread(image_path)
        if img is None:
            continue

        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        h_img, w_img = gray.shape

        # ลองตรวจจับใบหน้าด้วย multi-cascade
        faces = detect_faces_multi(gray)

        if len(faces) > 0:
            for (x, y, w, h) in faces:
                # Smart Padding 5%
                pad_x = int(0.05 * w)
                pad_y = int(0.05 * h)
                x1 = max(0, x - pad_x)
                y1 = max(0, y - pad_y)
                x2 = min(w_img, x + w + pad_x)
                y2 = min(h_img, y + h + pad_y)

                face_roi = gray[y1:y2, x1:x2]
                if face_roi.size == 0:
                    continue

                face_proc = preprocess_face(face_roi)
                faces_raw.append(face_proc)
                labels_raw.append(current_id)
                count += 1
        else:
            # Fallback: ใช้ภาพทั้งหมดเป็นใบหน้า (ภาพอาจเป็น face crop อยู่แล้ว)
            face_proc = preprocess_face(gray)
            faces_raw.append(face_proc)
            labels_raw.append(current_id)
            count += 1
            skipped += 1

    print(f"  [ID {current_id}] {person_name}: สกัดได้ {count} ภาพ (ใช้ fallback {skipped} ภาพ)")

print(f"\n✅ สกัดใบหน้ารวม: {len(faces_raw)} ภาพ")
print(f"📋 Label Dictionary: {label_dict}")

# ตรวจสอบจำนวนขั้นต่ำ
from collections import Counter
label_counts = Counter(labels_raw)
print(f"\n📊 จำนวนภาพต่อบุคคล:")
for lid, cnt in sorted(label_counts.items()):
    name = label_dict[lid]
    status = "✅" if cnt >= 10 else "⚠️ น้อยเกินไป"
    print(f"  {name}: {cnt} ภาพ {status}")

min_count = min(label_counts.values())
if min_count < 5:
    print(f"\n⚠️ คำเตือน: บุคคลบางคนมีภาพน้อยมาก ({min_count} ภาพ) ผลลัพธ์อาจไม่แม่นยำ")

In [ ]:
# 5. แบ่ง Train/Test + Data Augmentation
X_raw = np.array(faces_raw)
y_raw = np.array(labels_raw)

# แบ่งข้อมูล 80/20 แบบ Stratified
X_train_orig, X_test, y_train_orig, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42, stratify=y_raw
)

print(f"📊 ข้อมูลตั้งต้น -> Train: {len(X_train_orig)} | Test: {len(X_test)}")

def rotate_image(image, angle):
    h, w = image.shape[:2]
    center = (w // 2, h // 2)
    matrix = cv2.getRotationMatrix2D(center, angle, 1.0)
    return cv2.warpAffine(image, matrix, (w, h), borderMode=cv2.BORDER_REPLICATE)

def adjust_gamma(image, gamma=1.0):
    inv = 1.0 / gamma
    table = np.array([((i / 255.0) ** inv) * 255 for i in range(256)]).astype("uint8")
    return cv2.LUT(image, table)

X_train_aug = []
y_train_aug = []

for img, lbl in zip(X_train_orig, y_train_orig):
    # 1. ภาพต้นฉบับ
    X_train_aug.append(img)
    y_train_aug.append(lbl)

    # 2. พลิกซ้าย-ขวา
    X_train_aug.append(cv2.flip(img, 1))
    y_train_aug.append(lbl)

    # 3. หมุน ±5 องศา
    X_train_aug.append(rotate_image(img, 5))
    y_train_aug.append(lbl)
    X_train_aug.append(rotate_image(img, -5))
    y_train_aug.append(lbl)

    # 4. ปรับแสง
    X_train_aug.append(adjust_gamma(img, 0.8))
    y_train_aug.append(lbl)
    X_train_aug.append(adjust_gamma(img, 1.2))
    y_train_aug.append(lbl)

    # 5. เพิ่ม Gaussian noise เล็กน้อย
    noise = np.random.normal(0, 5, img.shape).astype(np.int16)
    noisy = np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8)
    X_train_aug.append(noisy)
    y_train_aug.append(lbl)

    # 6. หมุน ±10 องศา
    X_train_aug.append(rotate_image(img, 10))
    y_train_aug.append(lbl)
    X_train_aug.append(rotate_image(img, -10))
    y_train_aug.append(lbl)

X_train = np.array(X_train_aug)
y_train = np.array(y_train_aug)

print(f"🚀 Train หลัง Augmentation: {len(X_train)} ภาพ (เพิ่มขึ้น {len(X_train)//len(X_train_orig)} เท่า)")
print(f"🎯 Test สำหรับวัดผล: {len(X_test)} ภาพ")

In [ ]:
# 6. เทรนโมเดล LBPH + บันทึกโมเดลและ Config
recognizer = cv2.face.LBPHFaceRecognizer_create(
    radius=2, neighbors=16, grid_x=12, grid_y=12
)

print(f"กำลังฝึกสอนโมเดล LBPH ด้วย {len(X_train)} ภาพ...")
recognizer.train(X_train, y_train)
print("✅ ฝึกสอนโมเดลเสร็จสมบูรณ์!")

# --- หาโฟลเดอร์สำหรับบันทึก ---
save_dir = os.path.dirname(DATASET_PATH)  # บันทึกข้างๆ dataset เสมอ
if not os.path.exists(save_dir):
    save_dir = '.'

MODEL_FILE = 'Project_Ai_model_v2.yml'
CONFIG_FILE = 'model_config.json'

model_path = os.path.join(save_dir, MODEL_FILE)
config_path = os.path.join(save_dir, CONFIG_FILE)

recognizer.write(model_path)
print(f"💾 บันทึกโมเดลที่: {model_path}")

In [ ]:
# 7. สแกนหา Threshold ที่ดีที่สุดด้วย F1-Score
threshold_range = np.arange(25.0, 90.0, 0.5)
accuracy_scores = []
f1_scores = []

# ทำนายผล Test set ทั้งหมดครั้งเดียว
test_predictions = [recognizer.predict(img) for img in X_test]

for th in threshold_range:
    preds = [pred_id if dist <= th else -1 for pred_id, dist in test_predictions]
    acc = accuracy_score(y_test, preds)
    f1 = f1_score(y_test, preds, average='macro', zero_division=0)
    accuracy_scores.append(acc)
    f1_scores.append(f1)

# ใช้ F1-score เป็นเกณฑ์หลัก (ดีกว่า Accuracy สำหรับ imbalanced data)
best_idx = np.argmax(f1_scores)
OPTIMAL_THRESHOLD = float(threshold_range[best_idx])
best_f1 = f1_scores[best_idx]
best_acc = accuracy_scores[best_idx]

print(f"🏆 Optimal Threshold: {OPTIMAL_THRESHOLD:.1f}")
print(f"🌟 Best F1-Score: {best_f1 * 100:.2f}%")
print(f"🎯 Accuracy ที่ threshold นี้: {best_acc * 100:.2f}%")

# --- บันทึก Config ---
config = {
    "label_dict": {str(k): v for k, v in label_dict.items()},
    "optimal_threshold": OPTIMAL_THRESHOLD,
    "model_file": MODEL_FILE,
    "training_date": datetime.now().strftime("%Y-%m-%d %H:%M"),
    "total_train_samples": int(len(X_train)),
    "total_test_samples": int(len(X_test)),
    "best_f1_score": round(best_f1, 4),
    "best_accuracy": round(best_acc, 4),
    "person_names": person_names,
    "target_size": list(TARGET_SIZE)
}

with open(config_path, 'w', encoding='utf-8') as f:
    json.dump(config, f, ensure_ascii=False, indent=2)
print(f"💾 บันทึก Config ที่: {config_path}")

# --- วาดกราฟ ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4))

ax1.plot(threshold_range, [a*100 for a in accuracy_scores], label='Accuracy', color='blue', linewidth=2)
ax1.axvline(OPTIMAL_THRESHOLD, color='red', linestyle='--', label=f'Optimal ({OPTIMAL_THRESHOLD:.1f})')
ax1.set_title("Threshold vs Accuracy")
ax1.set_xlabel("Distance Threshold")
ax1.set_ylabel("Accuracy (%)")
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(threshold_range, [f*100 for f in f1_scores], label='F1-Score (macro)', color='green', linewidth=2)
ax2.axvline(OPTIMAL_THRESHOLD, color='red', linestyle='--', label=f'Optimal ({OPTIMAL_THRESHOLD:.1f})')
ax2.set_title("Threshold vs F1-Score")
ax2.set_xlabel("Distance Threshold")
ax2.set_ylabel("F1-Score (%)")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 8. ประเมินผล + Confusion Matrix
y_pred = []
confidences = []

for pred_id, dist in test_predictions:
    if dist <= OPTIMAL_THRESHOLD:
        y_pred.append(pred_id)
    else:
        y_pred.append(-1)
    confidences.append(dist)

y_pred = np.array(y_pred)
target_names = [label_dict[i] for i in range(len(label_dict))]

print("=" * 60)
print(f"🎯 Accuracy: {accuracy_score(y_test, y_pred) * 100:.2f}%")
print(f"📊 F1-Score (macro): {f1_score(y_test, y_pred, average='macro', zero_division=0) * 100:.2f}%")
print(f"📏 Distance เฉลี่ย: {np.mean(confidences):.2f} (Min: {np.min(confidences):.2f}, Max: {np.max(confidences):.2f})")
print("=" * 60)

print("\n📑 Classification Report:")
print(classification_report(y_test, y_pred, target_names=target_names,
                            labels=list(label_dict.keys()), zero_division=0))

plt.figure(figsize=(8, 6))
cm = confusion_matrix(y_test, y_pred, labels=list(label_dict.keys()))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=target_names, yticklabels=target_names)
plt.title(f"Confusion Matrix (Threshold: {OPTIMAL_THRESHOLD:.1f})")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.show()

In [ ]:
# 9. ทดสอบ YOLOv8 + LBPH Pipeline
print("กำลังโหลดโมเดล YOLOv8...")
yolo_model = YOLO('yolov8n.pt')
print("✅ โหลดโมเดล YOLOv8 สำเร็จ!")

def calculate_confidence_pct(distance, threshold):
    """คำนวณ Confidence % แบบ intuitive"""
    if distance <= 0:
        return 100.0
    if distance >= threshold:
        return 0.0
    return max(0.0, (1.0 - distance / threshold) * 100.0)

def recognize_pipeline(image_path, confidence_threshold=OPTIMAL_THRESHOLD):
    """Pipeline: YOLOv8 -> Haar Cascade -> CLAHE -> LBPH"""
    img = cv2.imread(image_path)
    if img is None:
        print(f"❌ ไม่พบไฟล์ภาพ: {image_path}")
        return

    display_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    h_img, w_img = img.shape[:2]

    # 1. YOLO ตรวจจับบุคคล (class 0: person)
    results = yolo_model(img, verbose=False, classes=[0])

    detected_faces = 0
    for r in results:
        for box in r.boxes:
            bx1, by1, bx2, by2 = map(int, box.xyxy[0])
            bx1, by1 = max(0, bx1), max(0, by1)
            bx2, by2 = min(w_img, bx2), min(h_img, by2)

            person_roi_gray = gray[by1:by2, bx1:bx2]
            if person_roi_gray.size == 0:
                continue

            faces = detect_faces_multi(person_roi_gray)

            for (fx, fy, fw, fh) in faces:
                pad_x = int(0.05 * fw)
                pad_y = int(0.05 * fh)
                ax1 = max(0, bx1 + fx - pad_x)
                ay1 = max(0, by1 + fy - pad_y)
                ax2 = min(w_img, bx1 + fx + fw + pad_x)
                ay2 = min(h_img, by1 + fy + fh + pad_y)

                face_crop = gray[ay1:ay2, ax1:ax2]
                if face_crop.size == 0:
                    continue

                face_proc = preprocess_face(face_crop)
                pred_id, dist = recognizer.predict(face_proc)
                conf_pct = calculate_confidence_pct(dist, confidence_threshold)

                if dist <= confidence_threshold:
                    name = label_dict.get(pred_id, "Unknown")
                    color = (0, 255, 0)
                    label_text = f"{name} {conf_pct:.0f}% (D:{dist:.1f})"
                else:
                    name = "Unknown"
                    color = (255, 0, 0)
                    label_text = f"{name} (D:{dist:.1f})"

                cv2.rectangle(display_img, (ax1, ay1), (ax2, ay2), color, 3)
                bw = int(len(label_text) * 12)
                cv2.rectangle(display_img, (ax1, max(0, ay1-25)), (ax1+bw, max(0, ay1)), color, -1)
                cv2.putText(display_img, label_text, (ax1+5, max(15, ay1-7)),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)
                detected_faces += 1

    # Fallback: ถ้า YOLO ไม่พบคน
    if detected_faces == 0:
        faces_fb = detect_faces_multi(gray)
        for (x, y, w, h) in faces_fb:
            pad_x, pad_y = int(0.05*w), int(0.05*h)
            x1, y1 = max(0, x-pad_x), max(0, y-pad_y)
            x2, y2 = min(w_img, x+w+pad_x), min(h_img, y+h+pad_y)
            face_crop = gray[y1:y2, x1:x2]
            if face_crop.size == 0:
                continue
            face_proc = preprocess_face(face_crop)
            pred_id, dist = recognizer.predict(face_proc)
            conf_pct = calculate_confidence_pct(dist, confidence_threshold)
            if dist <= confidence_threshold:
                name = label_dict.get(pred_id, "Unknown")
                color = (0, 255, 0)
                label_text = f"{name} {conf_pct:.0f}% (D:{dist:.1f})"
            else:
                name = "Unknown"
                color = (255, 0, 0)
                label_text = f"{name} (D:{dist:.1f})"
            cv2.rectangle(display_img, (x1, y1), (x2, y2), color, 3)
            bw = int(len(label_text)*12)
            cv2.rectangle(display_img, (x1, max(0,y1-25)), (x1+bw, max(0,y1)), color, -1)
            cv2.putText(display_img, label_text, (x1+5, max(15,y1-7)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, (255,255,255), 2)
            detected_faces += 1

    print(f"🔍 ตรวจพบ {detected_faces} ใบหน้า")
    plt.figure(figsize=(10, 8))
    plt.imshow(display_img)
    plt.title(f"YOLOv8 + LBPH (Threshold: {confidence_threshold:.1f})")
    plt.axis('off')
    plt.show()

# ทดสอบด้วยการสุ่มภาพจากโฟลเดอร์ test
import os
import random

possible_test_dirs = [
    'test',
    '/content/drive/MyDrive/Project_Ai/test'
]

TEST_DIR = None
for d in possible_test_dirs:
    if os.path.exists(d):
        TEST_DIR = d
        break

if TEST_DIR is None:
    print("❌ ไม่พบโฟลเดอร์ test")
else:
    test_images = [f for f in os.listdir(TEST_DIR) if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    if len(test_images) > 0:
        random_img = random.choice(test_images)
        sample_path = os.path.join(TEST_DIR, random_img)
        print(f"🎲 ระบบสุ่มได้รูปภาพ: {random_img}")
        print(f"🧪 กำลังทดสอบ Pipeline...")
        recognize_pipeline(sample_path)
    else:
        print(f"⚠️ ไม่พบไฟล์รูปภาพใดๆ ในโฟลเดอร์ {TEST_DIR}")